In [1]:
?fixest


No documentation for ‘fixest’ in specified packages and libraries:
you could try ‘??fixest’

In [2]:
library(fixest)


In [3]:
feols


function (fml, data, vcov, weights, offset, subset, split, fsplit, 
    split.keep, split.drop, cluster, se, ssc, panel.id, panel.time.step = NULL, 
    panel.duplicate.method = "none", fixef, fixef.rm = "perfect_fit", 
    fixef.tol = 1e-06, fixef.iter = 10000, fixef.algo = NULL, 
    collin.tol = 1e-09, nthreads = getFixest_nthreads(), lean = FALSE, 
    verbose = 0, warn = TRUE, notes = getFixest_notes(), only.coef = FALSE, 
    data.save = FALSE, fixef.keep_names = NULL, demeaned = FALSE, 
    mem.clean = FALSE, only.env = FALSE, env, ...) 
{
    dots = list(...)
    fromGLM = FALSE
    skip_fixef = FALSE
    if ("fromGLM" %in% names(dots)) {
        fromGLM = TRUE
        X = dots$X
        y = as.vector(dots$y)
        init = dots$means
        correct_0w = dots$correct_0w
        only.coef = FALSE
        IN_MULTI = FALSE
        if (verbose) {
            time_start = proc.time()
            show_timing = function(x, nl = TRUE) {
                cat(sfill(x, 20), ": ", -(t0 - (t0 <<- proc.time()))[3], 
                  "s", ifelse(nl, "\n", ""), sep = "")
            }
            t0 = proc.time()
        }
    }
    else {
        time_start = proc.time()
        show_timing = function(x, nl = TRUE) {
            cat(sfill(x, 20), ": ", -(t0 - (t0 <<- proc.time()))[3], 
                "s", ifelse(nl, "\n", ""), sep = "")
        }
        t0 = proc.time()
        if (missing(env)) {
            set_defaults("fixest_estimation")
            call_env = new.env(parent = parent.frame())
            env = try(fixest_env(fml = fml, data = data, weights = weights, 
                offset = offset, subset = subset, split = split, 
                fsplit = fsplit, split.keep = split.keep, split.drop = split.drop, 
                vcov = vcov, cluster = cluster, se = se, ssc = ssc, 
                panel.id = panel.id, panel.time.step = panel.time.step, 
                panel.duplicate.method = panel.duplicate.method, 
                fixef = fixef, fixef.rm = fixef.rm, fixef.tol = fixef.tol, 
                fixef.iter = fixef.iter, fixef.algo = fixef.algo, 
                collin.tol = collin.tol, nthreads = nthreads, 
                lean = lean, verbose = verbose, warn = warn, 
                notes = notes, only.coef = only.coef, data.save = data.save, 
                fixef.keep_names = fixef.keep_names, demeaned = demeaned, 
                mem.clean = mem.clean, origin = "feols", mc_origin = match.call(), 
                call_env = call_env, ...), silent = TRUE)
        }
        else if ((r <- !is.environment(env)) || !isTRUE(env$fixest_env)) {
            stop("Argument 'env' must be an environment created by a fixest estimation. Currently it is not ", 
                ifelse(r, "an", "a 'fixest'"), " environment.")
        }
        if (inherits(env, "try-error")) {
            err_msg = format_error_msg(env, "feols")
            stopi("{err_msg}")
        }
        check_arg(only.env, "logical scalar")
        if (only.env) {
            return(env)
        }
        y = get("lhs", env)
        X = get("linear.mat", env)
        nthreads = get("nthreads", env)
        init = 0
        if (!is.null(dots$X_demean)) {
            skip_fixef = TRUE
            X_demean = dots$X_demean
            y_demean = dots$y_demean
        }
        offset = get("offset.value", env)
        isOffset = length(offset) > 1
        if (isOffset) {
            if (is.list(y)) {
                for (i in seq_along(y)) {
                  y[[i]] = y[[i]] - offset
                }
            }
            else {
                y = y - offset
            }
        }
        weights = get("weights.value", env)
        isWeight = length(weights) > 1
        correct_0w = FALSE
        mem.clean = get("mem.clean", env)
        demeaned = get("demeaned", env)
        warn = get("warn", env)
        only.coef = get("only.coef", env)
        IN_MULTI = get("IN_MULTI", env)
        is_multi_root = get("is_multi_root", env)
        if (is_mul